# 中文新聞主題多分類 — word2vec + BiLSTM (Colab)

在 Colab(GPU)上訓練 word2vec 詞向量並用 BiLSTM 做新聞主題分類。
資料來源:自爬中央社 CNA 6 大分類新聞。

## 1. 安裝套件並 clone 課程 repo
TensorFlow 已內建於 Colab,只需補裝 jieba 與 gensim。

In [ ]:
!pip -q install jieba gensim
!git clone https://github.com/Upisofsht/Nlp2026.git
%cd Nlp2026/final_project
# 若 data/news_dataset.csv 未隨 repo 附上(被 .gitignore),
# 請改用左側檔案面板手動上傳到 final_project/data/ 後再繼續。

## 2. 載入資料 + jieba 斷詞

In [ ]:
import sys, pandas as pd
sys.path.insert(0, '.')
from src.preprocess import tokenize, load_stopwords

df = pd.read_csv('data/news_dataset.csv')
df['text'] = df['title'].fillna('') + ' ' + df['content'].fillna('')
stop = load_stopwords('stopwords.txt')
df['tokens'] = df['text'].apply(lambda t: tokenize(t, stop))
print(df['category'].value_counts())

## 3. 標籤編碼 + 切分 train/test

In [ ]:
from sklearn.model_selection import train_test_split
labels = sorted(df['category'].unique())
label2idx = {l: i for i, l in enumerate(labels)}
df['y'] = df['category'].map(label2idx)
tr, te = train_test_split(df, test_size=0.2, stratify=df['y'], random_state=42)
print('train', len(tr), 'test', len(te))

## 4. 訓練 word2vec + 轉成序列

In [ ]:
from src.features import (train_word2vec, build_vocab, texts_to_sequences,
                          build_embedding_matrix)
VEC, MAXLEN = 100, 200
w2v = train_word2vec(list(tr['tokens']), vector_size=VEC, min_count=2, epochs=10)
word2idx = build_vocab(list(tr['tokens']))
emb = build_embedding_matrix(word2idx, w2v, VEC)
X_tr = texts_to_sequences(list(tr['tokens']), word2idx, MAXLEN)
X_te = texts_to_sequences(list(te['tokens']), word2idx, MAXLEN)
y_tr, y_te = tr['y'].values, te['y'].values
print('vocab', len(word2idx), 'X_tr', X_tr.shape)

## 5. 建立並訓練 BiLSTM

In [ ]:
from src.models import build_bilstm
model = build_bilstm(len(word2idx), len(labels), embedding_matrix=emb,
                     vector_size=VEC, maxlen=MAXLEN)
model.summary()
model.fit(X_tr, y_tr, validation_split=0.1, epochs=10, batch_size=64)

## 6. 評估 + 對比 + 混淆矩陣

In [ ]:
import numpy as np
from src.evaluate import evaluate_predictions, plot_confusion_matrix
pred_idx = model.predict(X_te).argmax(axis=1)
pred = [labels[i] for i in pred_idx]
true = [labels[i] for i in y_te]
m = evaluate_predictions(true, pred)
print('BiLSTM:', f"acc={m['accuracy']:.4f} macroF1={m['macro_f1']:.4f}")
print(m['report'])
plot_confusion_matrix(true, pred, labels, out_path='confusion_bilstm.png')